# お土産お菓子研究所｜街頭インタビュー キーフレーム生成（ComfyUI on Colab）

このノートブックは **ComfyUI を Google Colab の無料GPU（T4）で起動**し、
人物リファレンスシートの顔を **IPAdapter FaceID PlusV2** で固定したまま、
**夏服・銀座ロケのキーフレーム画像**を生成します。

**パイプライン全体像**
```
[このノートブック] ComfyUI + IPAdapter FaceID → 夏服キーフレーム (9:16)
        ↓ ダウンロード → Higgsfield にアップロード
[Claude セッション] MiniMax H3 (image-to-video) → インタビュー動画化
```

**使い方**: ランタイム → すべてのセルを実行（GPUランタイム必須）。初回はモデルDLで15分前後かかります。


In [ ]:
#@title 1. GPU確認 & ComfyUI セットアップ
!nvidia-smi -L
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# IPAdapter Plus（FaceID対応カスタムノード）と依存
%cd /content/ComfyUI/custom_nodes
!git clone --depth 1 https://github.com/cubiq/ComfyUI_IPAdapter_plus.git
!pip install -q insightface onnxruntime-gpu
%cd /content/ComfyUI


In [ ]:
#@title 2. モデルダウンロード（SDXL + IPAdapter FaceID PlusV2）
import os
def dl(url, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        !wget -q --show-progress -O "{dst}" "{url}"

M = "/content/ComfyUI/models"
dl("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
   f"{M}/checkpoints/sd_xl_base_1.0.safetensors")
dl("https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin",
   f"{M}/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin")
dl("https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors",
   f"{M}/loras/ip-adapter-faceid-plusv2_sdxl_lora.safetensors")
dl("https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors",
   f"{M}/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors")
print("models ready")


In [ ]:
#@title 3. 人物リファレンス取得（リポジトリから）
%cd /content
!git clone --depth 1 -b claude/souvenir-sweets-interview-video-t4gvqb \
  https://github.com/hinokunijapankumamoto2-design/Omiyage-okashi-knowledge.git repo
!mkdir -p /content/ComfyUI/input
!cp repo/assets/video-refs/refA_face.jpg repo/assets/video-refs/refB_face.jpg /content/ComfyUI/input/
# ローカルの原寸シートを使う場合は、左パネルから直接 /content/ComfyUI/input/ にアップロードして
# 下の REFS のファイル名を差し替えてください（原寸のほうが同一性は上がります）


In [ ]:
#@title 4. ComfyUI をバックグラウンド起動
import subprocess, time, urllib.request
proc = subprocess.Popen(["python", "main.py", "--dont-print-server"], cwd="/content/ComfyUI")
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/queue"); print("ComfyUI ready"); break
    except Exception:
        time.sleep(2)


In [ ]:
#@title 5. 夏服キーフレーム生成（顔はFaceIDで固定）
import json, urllib.request, time

REFS = {
  "A": {"file": "refA_face.jpg",
        "outfit": "white sleeveless summer blouse, light beige flared skirt, small pastel handheld fan"},
  "B": {"file": "refB_face.jpg",
        "outfit": "navy sleeveless summer top, light grey pleated skirt, small black shoulder bag"},
}
SCENE = ("photorealistic candid street interview keyframe, young japanese woman standing on a "
         "Ginza sidewalk in Tokyo, midsummer bright afternoon, facing camera, black handheld "
         "interview microphone at lower left of frame, blurred department store facades and "
         "pedestrians in background, shallow depth of field, 35mm lens, natural skin texture")
NEG = "text, watermark, logo, subtitles, cartoon, anime, cgi, deformed hands, extra fingers, winter clothes, coat"

def workflow(ref_file, outfit, seed):
    return {
      "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"}},
      "2": {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0],
             "lora_name": "ip-adapter-faceid-plusv2_sdxl_lora.safetensors", "strength_model": 0.6}},
      "3": {"class_type": "IPAdapterUnifiedLoaderFaceID", "inputs": {"model": ["2", 0],
             "preset": "FACEID PLUS V2", "lora_strength": 0.0, "provider": "CUDA"}},
      "4": {"class_type": "LoadImage", "inputs": {"image": ref_file}},
      "5": {"class_type": "IPAdapterFaceID", "inputs": {"model": ["3", 0], "ipadapter": ["3", 1],
             "image": ["4", 0], "weight": 1.0, "weight_faceidv2": 1.5, "weight_type": "linear",
             "combine_embeds": "concat", "start_at": 0.0, "end_at": 1.0, "embeds_scaling": "V only"}},
      "6": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["1", 1], "text": f"{SCENE}, wearing {outfit}"}},
      "7": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["1", 1], "text": NEG}},
      "8": {"class_type": "EmptyLatentImage", "inputs": {"width": 832, "height": 1216, "batch_size": 1}},
      "9": {"class_type": "KSampler", "inputs": {"model": ["5", 0], "positive": ["6", 0], "negative": ["7", 0],
             "latent_image": ["8", 0], "seed": seed, "steps": 28, "cfg": 6.0,
             "sampler_name": "dpmpp_2m", "scheduler": "karras", "denoise": 1.0}},
      "10": {"class_type": "VAEDecode", "inputs": {"samples": ["9", 0], "vae": ["1", 2]}},
      "11": {"class_type": "SaveImage", "inputs": {"images": ["10", 0], "filename_prefix": "keyframe_summer"}},
    }

for name, ref in REFS.items():
    for seed in (101, 202):
        p = json.dumps({"prompt": workflow(ref["file"], ref["outfit"], seed)}).encode()
        req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=p,
                                     headers={"Content-Type": "application/json"})
        print(name, seed, urllib.request.urlopen(req).read().decode())

# キュー完了待ち
while True:
    q = json.load(urllib.request.urlopen("http://127.0.0.1:8188/queue"))
    if not q["queue_running"] and not q["queue_pending"]: break
    time.sleep(5)
print("done -> /content/ComfyUI/output/")


In [ ]:
#@title 6. 生成結果をZIPでダウンロード
!cd /content/ComfyUI/output && zip -q -r /content/keyframes_summer.zip .
from google.colab import files
files.download("/content/keyframes_summer.zip")


## 次のステップ（動画化）

1. 生成されたキーフレームから良いカットを選ぶ
2. Higgsfield（higgsfield.ai）にアップロード、または Claude セッションに URL を渡す
3. Claude セッション側で **MiniMax H3** の `start_image` / `image_references` に指定して動画化
   - テスト仕様: 10秒 / 9:16 / 2K（約40クレジット）
   - 台本: 「SNSで見て買うのは、もちもち系とか、映えるやつですね」→「でも、お土産で選ぶときは全然違う。結局、定番にしちゃう」

> **メモ**: Comfy Cloud（MCP直結）は有料サブスクリプション加入で partner ノードが解放され、
> このノートブックの工程を MCP から直接実行する構成に置き換えられます。
